<a href="https://githubtocolab.com/geonextgis/smartreview/blob/main/docs/examples/example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# SmartReview

This notebook walks through the recommended `smartreview` workflow:

1. Set the `OPENAI_API_KEY` environment variable (or use a `.env` file).
2. Run the high-level `rank_papers` pipeline on a Web of Science export.
3. Inspect the ranked DataFrame and the files written to disk.
4. Optionally drop down to the lower-level API for custom analysis (similarity histograms, percentile thresholds, BibTeX, etc.).

Place your Web of Science `.xls`/`.xlsx` export in `docs/examples/data/` before running.

## 1. Setup

In [ ]:
# Uncomment to install the package and its runtime dependencies:
# %pip install smartreview
# %pip install python-dotenv matplotlib

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt

from smartreview import (
    rank_papers,
    load_wos_export,
    create_openai_client,
    EmbeddingCache,
    embed_papers,
    get_embedding,
    calculate_cosine_similarity,
    get_top_k_papers,
    top_k_by_percentile,
    create_top_k_dataframe,
    save_top_k_papers,
    generate_bibtex_file,
    print_top_k_summary,
)

load_dotenv()
print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

## 2. Define your research interests

In [ ]:
interest_text = """
I am interested in research at the intersection of machine learning, deep learning, and artificial intelligence applied to agriculture and crop yield prediction.
This includes methods such as LSTM, RNN, CNN, transformers, XGBoost, gradient boosting, and hybrid or process-based models like DSSAT or APSIM.
I am also interested in studies using remote sensing and Earth observation data, including Sentinel, Landsat, MODIS, NDVI, and EVI,
as well as climate change impacts such as drought, heat stress, and extreme events on crops.
Phenology, growing season analysis, crop calendars, and time series modeling are also relevant.
Overall, the focus is on improving food production systems, crop modeling, and global agricultural assessments.
""".strip()

INPUT_FILE = '/beegfs/halder/GITHUB/PROJECT/smartreview/docs/examples/test_data.xls'
OUTPUT_DIR = 'data'

## 3. `rank_papers`

`rank_papers` reads the export, embeds papers + interest, ranks by cosine similarity, and writes CSV / Excel / BibTeX to `OUTPUT_DIR`. Embeddings are cached on disk under `<OUTPUT_DIR>/embeddings/cache/`, so re-running this cell after editing your interest text will re-use paper embeddings and only re-embed the interest statement.

In [ ]:
result = rank_papers(
    input_path=INPUT_FILE,
    interest_text=interest_text,
    output_dir=OUTPUT_DIR,
    top_percentile=80.0,  # keep the top 20%; use top_k=N for a fixed count
)

df = result['dataframe']
print(f"Selected {len(df)} papers.")
print(f"  CSV:    {result['files']['csv']}")
print(f"  Excel:  {result['files']['excel']}")
print(f"  BibTeX: {result['bibtex']['file']}")

In [ ]:
print_top_k_summary(df, k=len(df), show_rows=10)

## 4. Visualise the similarity distribution

In [ ]:
scores = np.array([s for _, s in result['similarities']])
threshold = result['top_k'][-1][1] if result['top_k'] else float('nan')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores, bins=40, color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(threshold, color='red', linestyle='--', linewidth=2,
           label=f'Selection threshold: {threshold:.3f}')
ax.set_xlabel('Cosine similarity to interest statement')
ax.set_ylabel('Number of papers')
ax.set_title('Similarity score distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Lower-level API (optional)

If you need finer control — e.g. custom text preprocessing, multiple interest queries against the same corpus, or a different embedding model — you can call the building blocks directly. The on-disk cache means you only pay the API once per (paper, model) pair.

In [ ]:
client = create_openai_client()
cache = EmbeddingCache('data/embeddings/cache')

data, summary = load_wos_export(INPUT_FILE)
paper_embeddings = embed_papers(summary, client=client, cache=cache)
interest_embedding = get_embedding(interest_text, client=client, cache=cache)

similarities = calculate_cosine_similarity(interest_embedding, paper_embeddings)

# Either a fixed K...
top_50 = get_top_k_papers(similarities, k=50)
# ...or a percentile-based threshold:
top_20pct = top_k_by_percentile(similarities, percentile=80.0)

top_df = create_top_k_dataframe(top_50, data)
save_top_k_papers(top_df, output_dir='data', k=50)
generate_bibtex_file(top_df, output_dir='data', k=50)
print(f'Wrote top 50 (fixed K) and selected {len(top_20pct)} papers via top 20%.')

## 6. CLI alternative

Everything above is also available as a shell command after `pip install -e .`:

```bash
smartreview \
  --input data/Nature_Communications_\(2024-2026\).xls \
  --interest-file interest.txt \
  --output-dir data \
  --top-percentile 80
```